# 04 · Disease type: DR, MH and ODC (multi-label)

Predicts **Diabetic Retinopathy (DR)**, **Media Haze (MH)** and **Optic Disc Cupping (ODC)** as three independent yes/no outputs.

**What changed from the original Part 4 and Part 5 notebooks**

| Original | Problem | Now |
|---|---|---|
| Softmax over 3 classes | An eye with both DR and ODC could only get one label; 99 training images have 2+ of these | Three sigmoid outputs, each disease predicted independently |
| Only the 724 images sorted into DR/MH/ODC folders | About 150 target-disease images were lost to other folders; the model never saw healthy eyes or other diseases, so it could not learn to say "none of these" | All 1,920 training images; images without a target disease are negatives for all three |
| `X_train` computed once with InceptionV3 | The Xception, VGG16, MobileNetV2, EfficientNetB5 and SE-ResNeXt results were all InceptionV3 features with different heads | Each backbone is trained end to end on its own |
| Same shuffle/label misalignment as Part 3 | Accuracy 30-46%, below always predicting DR (47%) | Fixed via `tf.data` |
| Test set used for early stopping | Optimistic score | Validation set for early stopping and thresholds |
| Accuracy only | Hides per-disease performance | Per-disease AUC, PR-AUC, F1, sensitivity, specificity, with thresholds tuned on validation |

Runtime on a Colab T4: roughly 15-25 minutes per backbone.

In [ ]:
# --- Setup (run once per session) -------------------------------------------
# On Colab: mounts Google Drive, clones this repo and installs requirements.
# Locally: just makes the `iluminado` package importable.
import os, sys, subprocess

REPO_URL = "https://github.com/cathykam2006/Capstone-Project-Illuminado-Retinal--Image-Classification.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/repo"], check=True)
    os.chdir("/content/repo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

REPO_DIR = next(p for p in [os.getcwd(), os.path.dirname(os.getcwd())] if os.path.isdir(os.path.join(p, "iluminado")))
sys.path.insert(0, REPO_DIR)
print("Repo:", REPO_DIR)

In [ ]:
# --- Config: edit these paths ---------------------------------------------------
# PROJECT_DIR holds the cached images, trained models and results. On Colab it
# lives on your Google Drive so it survives between sessions.
PROJECT_DIR = "/content/drive/MyDrive/iluminado" if IN_COLAB else os.path.join(REPO_DIR, "work")
LABEL_DIR = os.path.join(REPO_DIR, "data")            # the three RFMiD label CSVs (in the repo)
MANIFEST = os.path.join(PROJECT_DIR, "manifest.csv")  # written by notebook 01
os.makedirs(PROJECT_DIR, exist_ok=True)

In [ ]:
# --- Training config ---------------------------------------------------------------
IMG_SIZE = 384        # fundus lesions are small; 384 keeps far more detail than the old 150-256
BATCH_SIZE = 16
BACKBONES = ["efficientnet_b0", "densenet121", "convnext_tiny"]   # different families -> more useful ensemble
HEAD_EPOCHS, FINETUNE_EPOCHS = 5, 30
SEED = 42

import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf, keras
keras.utils.set_random_seed(SEED)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus or "none (training will be very slow; in Colab use Runtime > Change runtime type > GPU)")
if gpus:
    keras.mixed_precision.set_global_policy("mixed_float16")   # ~2x faster on T4/A100

from iluminado.data import make_dataset, TARGET_DISEASES
from iluminado import models as M, evaluate as E
m = pd.read_csv(MANIFEST)
MODEL_DIR = os.path.join(PROJECT_DIR, "models"); os.makedirs(MODEL_DIR, exist_ok=True)
RESULTS_DIR = os.path.join(PROJECT_DIR, "results"); os.makedirs(RESULTS_DIR, exist_ok=True)

## Data

In [ ]:
TASK = "multilabel"
LABELS = TARGET_DISEASES
split = {s: m[m.split == s].reset_index(drop=True) for s in ("train", "val", "test")}
y = {s: split[s][LABELS].to_numpy("float32") for s in split}

train_ds = make_dataset(split["train"].path, y["train"], IMG_SIZE, BATCH_SIZE, training=True, seed=SEED)
val_ds = make_dataset(split["val"].path, y["val"], IMG_SIZE, BATCH_SIZE, training=False)
test_ds = make_dataset(split["test"].path, y["test"], IMG_SIZE, BATCH_SIZE, training=False)

pos_weight = M.positive_weights(y["train"])
pd.DataFrame({s: y[s].sum(axis=0).astype(int) for s in y}, index=LABELS).assign(pos_weight=pos_weight.round(2))

## Train each backbone

In [ ]:
probs = {"val": {}, "test": {}}
for bb in BACKBONES:
    path = os.path.join(MODEL_DIR, f"{TASK}_{bb}.keras")
    if os.path.exists(path):
        print(f"[{bb}] found saved model, skipping training")
        model = M.load(path)
    else:
        print(f"\n===== {bb} =====")
        keras.backend.clear_session()
        model = M.build_model(bb, IMG_SIZE, n_outputs=len(LABELS))
        h1, h2 = M.train(model, train_ds, val_ds, pos_weight, n_outputs=len(LABELS), ckpt_path=path,
                         head_epochs=HEAD_EPOCHS, finetune_epochs=FINETUNE_EPOCHS)
        model = M.load(path)
        pd.DataFrame(h2.history).to_csv(os.path.join(RESULTS_DIR, f"{TASK}_{bb}_history.csv"), index=False)
    for s, ds in (("val", val_ds), ("test", test_ds)):
        probs[s][bb] = E.predict(model, ds)
    np.save(os.path.join(RESULTS_DIR, f"{TASK}_{bb}_val.npy"), probs["val"][bb])
    np.save(os.path.join(RESULTS_DIR, f"{TASK}_{bb}_test.npy"), probs["test"][bb])

## Ensemble and per-disease thresholds (validation set only)

Each disease gets its own threshold, chosen to maximise F1 on validation.

In [ ]:
from sklearn.metrics import roc_auc_score
yv, yt = y["val"], y["test"]
val_auc = {bb: roc_auc_score(yv, p, average="macro") for bb, p in probs["val"].items()}
probs["val"]["ensemble"] = E.ensemble({k: probs["val"][k] for k in BACKBONES}, val_auc)
probs["test"]["ensemble"] = E.ensemble({k: probs["test"][k] for k in BACKBONES}, val_auc)
thresholds = {k: [E.threshold_for_f1(yv[:, j], p[:, j]) for j in range(len(LABELS))] for k, p in probs["val"].items()}
pd.Series(val_auc, name="validation macro AUC").round(3)

## Final test results (test set used once)

In [ ]:
summary = {}
for k, p in probs["test"].items():
    rep = E.multilabel_report(yt, p, thresholds[k], LABELS)
    summary[k] = rep.loc["macro avg", ["AUC", "PR-AUC", "F1", "sensitivity", "specificity"]]
summary = pd.DataFrame(summary).T
lo, hi = E.bootstrap_ci(yt, probs["test"]["ensemble"], metric=lambda a, b: roc_auc_score(a, b, average="macro"))
print(f"Ensemble macro AUC 95% CI: {lo:.3f}-{hi:.3f}")
summary.to_csv(os.path.join(RESULTS_DIR, f"{TASK}_test_summary.csv"))
summary.round(3)

In [ ]:
per_disease = E.multilabel_report(yt, probs["test"]["ensemble"], thresholds["ensemble"], LABELS)
per_disease.to_csv(os.path.join(RESULTS_DIR, f"{TASK}_ensemble_per_disease.csv"))
per_disease.round(3)

PR-AUC should be read against the prevalence column: for a rare disease, a random model's PR-AUC equals its prevalence, so 0.6 on a 6%-prevalence disease is a large improvement over chance.

In [ ]:
from sklearn.metrics import roc_curve
fig, axes = plt.subplots(1, len(LABELS), figsize=(5 * len(LABELS), 4.5))
for j, d in enumerate(LABELS):
    for k, p in probs["test"].items():
        fpr, tpr, _ = roc_curve(yt[:, j], p[:, j])
        axes[j].plot(fpr, tpr, lw=2.5 if k == "ensemble" else 1.1, label=f"{k} ({roc_auc_score(yt[:, j], p[:, j]):.3f})")
    axes[j].plot([0, 1], [0, 1], "k--", lw=1); axes[j].set_title(d); axes[j].legend(fontsize=7)
    axes[j].set(xlabel="1 - specificity", ylabel="sensitivity")
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, f"{TASK}_roc.png"), dpi=150); plt.show()

## Grad-CAM per disease

In [ ]:
from PIL import Image
from iluminado.gradcam import gradcam, overlay
best = max(BACKBONES, key=lambda k: val_auc[k])
model = M.load(os.path.join(MODEL_DIR, f"{TASK}_{best}.keras"))
fig, axes = plt.subplots(2, len(LABELS), figsize=(4 * len(LABELS), 8))
for j, d in enumerate(LABELS):
    row = split["test"][split["test"][d] == 1].sample(1, random_state=2).iloc[0]
    img = np.asarray(Image.open(row.path).convert("RGB").resize((IMG_SIZE, IMG_SIZE)), dtype="float32")
    p = model.predict(img[None], verbose=0)[0]
    axes[0, j].imshow(img.astype("uint8")); axes[0, j].set_title(f"#{row.ID}: true {d}", fontsize=9)
    axes[1, j].imshow(overlay(img, gradcam(model, img, output_index=j)))
    axes[1, j].set_title(" ".join(f"{n}={v:.2f}" for n, v in zip(LABELS, p)), fontsize=8)
    for a in axes[:, j]: a.axis("off")
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, f"{TASK}_gradcam.png"), dpi=150); plt.show()

## Save the ensemble config for the app

In [ ]:
config = {
    "task": TASK, "img_size": IMG_SIZE, "labels": LABELS,
    "models": {bb: f"{TASK}_{bb}.keras" for bb in BACKBONES},
    "weights": {bb: float(val_auc[bb]) for bb in BACKBONES},
    "thresholds": [float(t) for t in thresholds["ensemble"]],
    "threshold_rule": "max F1 per disease on validation set",
    "test_macro_auc": float(summary.loc["ensemble", "AUC"]),
}
with open(os.path.join(MODEL_DIR, f"{TASK}_ensemble.json"), "w") as f:
    json.dump(config, f, indent=2)
print(json.dumps(config, indent=2))

In [ ]:
# Markdown tables for the README
print(summary.round(3).to_markdown()); print(); print(per_disease.round(3).to_markdown())